# ML-09 — Validation and Research Claim Audit

**Lane 2 (refresh / opportunity scoring).** This is the closing honesty pass on everything before
it. I take two findings from the FlyRank research paper, check my own Week-5 model under an honest
split (before/after), re-run the leakage hunt on the final feature set, and rewrite my boldest
sentence in language the evidence can carry.

Built with the `hunting-leakage-and-validating` and `writing-honest-claims` skills.

**The one-line claim this notebook earns:** a naive random split and a deliberately leaky feature
would make my model look like a ~1.0-AUC miracle; the moment I use an honest client-grouped split
and features knowable only at decision time, the real skill is modest (AUC ~0.62, precision@20 ~0.74
against a 0.67 base rate). Validation design — not the model — is what decides whether a number is a
result or a confession.

## 1. Two paper findings + my methodology questions

I read the full paper (`docs/flyrank-seo-research-march-2026.pdf`, 36 pages, 341,701 content pieces
across 57 brands) and picked the two findings that bear most directly on my lane (refresh /
opportunity scoring), because both hinge on **content age** — my strongest feature.

For each I ask the two questions the skill insists on: *where does the label come from?* and *does
the validation design carry the claim?* Constructive tone: the paper is openly observational, and my
job is to make my own design stronger, not to score points.

### Finding #1 — The Anatomy of Growing Content (CONFIRMED)

Growing pages have a different structural profile: they are **~37.6% longer** (3.2K vs 2.3K words)
and **~20% younger** (184 vs 230 days). Declining content is *not* just "bad" — many declining pages
still carry meaningful impressions, but they are older and thinner.

- **Where the label comes from:** impression **direction** (`up` vs `down`), a 30d-vs-prior-30d
  impression change — the paper's `Trend Direction`.
- **Where the evidence lives:** a *cross-sectional* comparison of two already-outcome groups
  (74,187 rising vs 45,272 falling), averaged age and word count per group.
- **Does the design carry the claim?** It carries an **association**: "in this data, growing pages
  were longer and younger." It cannot separate *why* — a page may be young *and* growing, or a team
  may already refresh only the pages they expect to win (selection). No intervention was made, so it
  cannot support "make pages longer/younger and they will grow.

### Finding #2 — The Content Performance Curve (CONFIRMED)

Content **peaks at 61–90 days** (health 33.1), stabilizes on a **maturation plateau 91–180 days**,
then hits a **decay cliff at 271–365 days** (health drops to 14). The 365+ rebound (25.1) is
concentrated in *older pages that were refreshed* — and the paper itself warns this is **not**
evidence that age reverses decline on its own.

- **Where the label comes from:** a **health score** (FlyRank composite) bucketed by content age.
- **Where the evidence lives:** a cross-section of *age vs health* across the portfolio — different
  pages at different ages, not the same page tracked as it ages.
- **Does the design carry the claim?** It carries a **pattern across pages**, not a within-page
  lifecycle. The 365+ "recovery" group is **selected** (someone already chose to refresh those
  pages), so part of that rebound is the *choosing*, not the *age*. The paper states this honestly.

### My methodology questions to myself (from both)

1. **Confounding by selection:** if a portfolio team only refreshes promising pages, "young +
   growing" and "365+ rebound" both inherit that pick — my features must never encode who was chosen.
2. **Label / feature window separation:** the paper compares a page's health (or trend) in the same
   window it measures age. My design instead puts every feature in window **B** (knowable at
   decision time `t`) and the label only in window **F** (after `t`), so I never compare a page's
   current state to its own future outcome through a shared window.
3. **Cross-section vs holdout:** the paper's numbers are in-sample cross-sections. I repeat the
   "does it hold on a client it never saw?" test below (§2, §3) — the honest check the paper's
   descriptive comparisons can't do.

In [1]:
findings = {
    "F1": {
        "tag": "CONFIRMED — The Anatomy of Growing Content",
        "label": "Impression direction (up vs down), 30d vs prior-30d",
        "evidence": "Growing: 3.2K words, 184d age | Declining: 2.3K words, 230d age",
        "design": "Cross-sectional group comparison, n=74,187 rising vs 45,272 falling",
        "carry": "Association only. Selection confound unresolved; no intervention, no cause.",
    },
    "F2": {
        "tag": "CONFIRMED — The Content Performance Curve",
        "label": "Health score buckets by content age",
        "evidence": "Peak 61-90d (33.1); plateau 91-180d; cliff 271-365d (14); 365+ (25.1) only in refreshed pages",
        "design": "Cross-section of age vs health; 365+ rebound is a selected (refreshed) subset",
        "carry": "Pattern across pages. 365+ rebound confounded by which pages were chosen to refresh.",
    },
}
print("Two paper findings read, with label source, evidence, design, and what the claim can carry:\n")
for k, v in findings.items():
    print(k, "-", v["tag"])
    print("  label  :", v["label"])
    print("  evidence:", v["evidence"])
    print("  design :", v["design"])
    print("  carries:", v["carry"], "\n")

Two paper findings read, with label source, evidence, design, and what the claim can carry:

F1 - CONFIRMED — The Anatomy of Growing Content
  label  : Impression direction (up vs down), 30d vs prior-30d
  evidence: Growing: 3.2K words, 184d age | Declining: 2.3K words, 230d age
  design : Cross-sectional group comparison, n=74,187 rising vs 45,272 falling
  carries: Association only. Selection confound unresolved; no intervention, no cause. 

F2 - CONFIRMED — The Content Performance Curve
  label  : Health score buckets by content age
  evidence: Peak 61-90d (33.1); plateau 91-180d; cliff 271-365d (14); 365+ (25.1) only in refreshed pages
  design : Cross-section of age vs health; 365+ rebound is a selected (refreshed) subset
  carries: Pattern across pages. 365+ rebound confounded by which pages were chosen to refresh. 



## 2. My model under an honest split (before / after)

The same RF trained in Week 5 (ML-08) is re-run three ways on the final feature set:

- **BEFORE — naive random split.** The easy, tempting way. Rows split randomly; rows from one
  client leak into both train and test, so the model can partly memorize the client.
- **AFTER — honest grouped split.** `GroupKFold` by client: test folds are clients the model
  **never saw**. The only split that answers "does it work on a new client?".
- **LEAKY (trap) — random split but with `imp_f` added.** The label's own numerator. This is what a
  score looks like when a feature quietly reads the answer.

All three use the **same 22-column B-only feature set**; only the split (and one deliberately dirty
feature) differ. Base rate rides along, every time.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath("../scripts"))
import duckdb, pandas as pd, numpy as np
from datetime import timedelta
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
import hf_query

SEED = 1
np.random.seed(SEED)

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '" + hf_query.get_token() + "')")
REL = hf_query.REL
T = {
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
D_MAX = con.sql(f"SELECT MAX(report_date) FROM {T['daily']}").fetchone()[0]
t = D_MAX - timedelta(days=30); b_lo = t - timedelta(days=30)

feat = con.sql(f"""
WITH win AS (
    SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available
    FROM {T['daily']} WHERE month IN ('{t:%Y-%m}', '{D_MAX:%Y-%m}')
),
agg AS (
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_b,
           SUM(CASE WHEN report_date >  DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_f,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_clicks ELSE 0 END) AS clk_b,
           SUM(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available THEN 1 ELSE 0 END) AS gsc_days_b,
           AVG(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0 THEN gsc_avg_position END) AS pos_avg_b,
           STDDEV(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0 THEN gsc_avg_position END) AS pos_vol_b
    FROM win GROUP BY 1, 2
),
j AS (
    SELECT a.client_hash_id, a.content_hash_id, a.imp_b, a.imp_f, a.clk_b, a.gsc_days_b,
           a.pos_avg_b, a.pos_vol_b,
           c.content_type, c.word_count, c.char_count, c.keyword_char_count,
           c.keyword_token_count, c.url_char_count, c.main_intent, c.competition_level,
           c.category_count, c.search_volume, c.backlinks,
           c.content_created_date, c.content_updated_date
    FROM agg a LEFT JOIN {T['content']} c USING (client_hash_id, content_hash_id)
)
SELECT *, DATE '{t}' - content_created_date AS age_days,
       DATE '{t}' - content_updated_date AS days_since_update,
       CASE WHEN imp_f < 0.8 * imp_b THEN 1 ELSE 0 END AS declined_30d
FROM j WHERE imp_b >= 100 AND gsc_days_b >= 15
""").df()
feat["ctr_b"] = feat.clk_b / feat.imp_b
y = feat.declined_30d.values
print("eligible pages:", len(feat), "| base rate:", round(y.mean(), 4))

base_feats = ["imp_b", "clk_b", "gsc_days_b", "pos_avg_b", "pos_vol_b"]
count_feats = ["word_count", "char_count", "keyword_char_count", "keyword_token_count",
               "url_char_count", "category_count", "search_volume", "backlinks",
               "age_days", "days_since_update"]
legal = feat[base_feats + count_feats + ["ctr_b", "content_type", "main_intent", "competition_level"]].copy()
for c in ["word_count", "search_volume", "backlinks"]:
    legal["has_" + c] = (~feat[c].isna()).astype(int)

def encode(X):
    X = X.copy()
    for c in X.select_dtypes(include=["object", "str"]).columns:
        X[c] = LabelEncoder().fit_transform(X[c].astype(str))
    return X.fillna(0)

X = encode(legal)
print("feature matrix:", X.shape)

C:\Users\Bogdan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


eligible pages: 108254 | base rate: 0.6744


feature matrix: (108254, 22)


In [3]:
def p_at_k(s, l, k):
    o = np.argsort(-np.asarray(s))
    return np.asarray(l)[o[:k]].mean()

def rf_pipe():
    return RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)

# AFTER - honest: grouped by client (model never sees these clients)
gkf = GroupKFold(n_splits=5)
g_auc, g_p20 = [], []
for tri, tei in gkf.split(X, y, groups=feat.client_hash_id.values):
    p = rf_pipe().fit(X.iloc[tri], y[tri]).predict_proba(X.iloc[tei])[:, 1]
    g_auc.append(roc_auc_score(y[tei], p)); g_p20.append(p_at_k(p, y[tei], 20))

# BEFORE - naive random split (rows, not clients, split)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
r_auc, r_p20 = [], []
for _ in range(3):
    p = rf_pipe().fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    r_auc.append(roc_auc_score(yte, p)); r_p20.append(p_at_k(p, yte, 20))

# LEAKY trap - naive split but with imp_f (the label's numerator)
Xlk = X.copy(); Xlk["imp_f"] = feat.imp_f
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xlk, y, test_size=0.2, random_state=SEED, stratify=y)
p = rf_pipe().fit(Xtr2, ytr2).predict_proba(Xte2)[:, 1]
l_auc = roc_auc_score(yte2, p); l_p20 = p_at_k(p, yte2, 20)

print("base rate                                  :", round(y.mean(), 4))
print("BEFORE (naive random split)                : AUC", round(np.mean(r_auc), 3),
      "| p@20", round(np.mean(r_p20), 3))
print("AFTER  (honest grouped by client)          : AUC", round(np.mean(g_auc), 3),
      "| p@20", round(np.mean(g_p20), 3))
print("LEAKY  (+ imp_f, naive split - the trap)   : AUC", round(l_auc, 3),
      "| p@20", round(l_p20, 3))
print("honesty gap (BEFORE - AFTER AUC)           :", round(np.mean(r_auc) - np.mean(g_auc), 3))

base rate                                  : 0.6744
BEFORE (naive random split)                : AUC 0.79 | p@20 1.0
AFTER  (honest grouped by client)          : AUC 0.615 | p@20 0.74
LEAKY  (+ imp_f, naive split - the trap)   : AUC 0.991 | p@20 1.0
honesty gap (BEFORE - AFTER AUC)           : 0.175


**Reading (before / after):**

| run | split | feature set | AUC | p@20 |
|---|---|---|---|---|
| BEFORE | naive random | B-only | ~0.79 | ~1.00 |
| **AFTER** | **grouped by client** | **B-only** | **~0.62** | **~0.74** |
| LEAKY | naive random | B-only + `imp_f` | ~0.99 | ~1.00 |

The **leaky run is the cautionary tale**: one `F`-window feature and the score snaps to ~1.0 — that
would be a wasted, worthless "win." The **random-split run** (AUC ~0.79, p@20 ~1.00) looks far better
than the honest one (~0.62); that gap is the client-memorization a random split silently buys, and
p@20 ~1.00 on a held-out *random* fifth is a red flag, not a result. The **honest grouped run** (~0.62
AUC, p@20 ~0.74, base rate 0.67) is the number that would actually hold on a client the model was never
trained on. For a true *time* split there is only one snapshot (`t` is fixed), so the time-awareness
lives in the B/F window separation instead — features are knowable at `t`, labels come after.

This is the direct answer to section 1's question *does the validation design carry the claim?*: the
paper's descriptive cross-sections cannot, but this holdout design is what lets my own numbers claim
"ranks/flags … at precision@K" and nothing more.

## 3. Leakage audit on the final feature set

The same three-way hunt from Week 5 (ML-05), re-run on the **final** 22-column feature set to confirm
nothing new sneaked in: label-derived / future-window leaks, overlapping windows, and the split
honesty check. Base rate printed beside every metric. Deliberately adding each leak and watching the
score move is the confession signed by the data itself.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def pipe():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

def ev(df, seed=0):
    d = df.copy()
    for c in d.select_dtypes(include=["object", "str"]).columns:
        d[c] = LabelEncoder().fit_transform(d[c].astype(str))
    d = d.fillna(0)
    Xtr, Xte, ytr, yte = train_test_split(d, y, test_size=0.25, random_state=seed, stratify=y)
    return roc_auc_score(yte, pipe().fit(Xtr, ytr).predict_proba(Xte)[:, 1])

print("base rate                             :", round(y.mean(), 4))
print("final set (B-only + static)           : AUC", round(ev(legal), 3))
leak = legal.copy(); leak["imp_f"] = feat.imp_f
print("+ imp_f (label-derived, F window)     : AUC", round(ev(leak), 3))
fut = legal.copy(); fut["fwd_imp_signal"] = (feat.imp_f > feat.imp_b).astype(int)
print("+ future-window signal (straddles t)  : AUC", round(ev(fut), 3))
prod = legal.copy()
prod["provider_model"] = (feat.imp_b * 0 + 1)  # placeholder, excluded on principle (see note)
print("grouped-by-client (honest, from +2)    : AUC", round(np.mean(g_auc), 3))

base rate                             : 0.6744


final set (B-only + static)           : AUC 0.63


+ imp_f (label-derived, F window)     : AUC 0.997


+ future-window signal (straddles t)  : AUC 0.869
grouped-by-client (honest, from +2)    : AUC 0.615


**Reading (leakage audit):**

- The **final legal set** (~0.63) is barely above the 0.67 base rate — honest, unimpressive, and true.
- The moment `imp_f` (the label's own numerator, from window F) is added, AUC → **~1.0**. Near-perfect
  score = confession, not win.
- A **future-window signal** that straddles `t` lifts AUC to ~0.87 — future information again.
- **Product flags** (`provider_used` / `model_used`) are excluded on *principle* (decision-derived,
  circular) and on *privacy* (they can carry non-anonymized free-text). We do not read or print their
  values here. They could only ever be a baseline to beat, never inputs.

**Excluded for the same reasons as ML-05 (final set):** `imp_f` & any `F`-derived aggregate; the whole
`fact_content_query_90d` table (fixed 90-day window straddles `t`); `provider_used` / `model_used`;
`ga4_*` / `sessions_*` / `ai_*` / `scroll_events` (zero-filled ~74% off GA4); product-workflow
timestamps; high-cardinality hash IDs; `is_deleted` / `is_published` (population filters only);
`client_hash_id` / `content_hash_id` (grouping/splitting only, never features).

## 4. Claim rewrite

**My own boldest sentence** (from the ML-07 / ML-08 work):

> "Our model predicts which pages will decline, so an editor can fix the right pages first."

That is a **causal-sounding** sentence. This portfolio is one cross-sectional snapshot with no
intervention — by the `writing-honest-claims` ladder I may *observe*, *associate*, and *say the
validated model ranks*, but I must never claim fixing a page will make it recover. Below I walk the
sentence down to language the evidence can actually carry.

In [5]:
bold = (
    "Our model predicts which pages will decline, "
    "so an editor can fix the right pages first."
)
ladder = [
    ("Observed (this data)",
     "In this portfolio snapshot, ~67% of eligible pages met the declined_30d condition, and "
     "pages off page one declined more often than those on it (ML-06)."),
    ("Measured / associated",
     "Older pages in the 6-12 month band showed the highest decline share; length showed no "
     "clear monotonic association with traffic."),
    ("Validated model (holdout)",
     "On a client-grouped holdout, the Random Forest ranks likely-decliners at precision@20 ~0.70 "
     "versus the 0.67 base rate (AUC ~0.61)."),
    ("Decision-support (no cause)",
     "These pages look worth an editor reviewing first, because the pattern in this data is "
     "associated with decline - not a promise that refreshing them will restore rankings."),
]
print("BOLD ORIGINAL:", bold, "\n")
for tier, text in ladder:
    print(f"[{tier}]")
    print("  " + text + "\n")

print("Skeptic check - words that exceeded the evidence in the original:")
for w in ["predict", "so an editor can fix", "decline"]:
    print("  '{0}': only allowed as '{0} ranks/flags ...' (model), or 'associated with' (data), "
          "never 'fixing causes'.".format(w))
print("\nRewrite (decision-support):")
print("  'On client-grouped holdout the model ranks likely-decliners first at p@20 ~0.70 (base "
      "0.67); these pages are worth an editor reviewing first.'")

BOLD ORIGINAL: Our model predicts which pages will decline, so an editor can fix the right pages first. 

[Observed (this data)]
  In this portfolio snapshot, ~67% of eligible pages met the declined_30d condition, and pages off page one declined more often than those on it (ML-06).

[Measured / associated]
  Older pages in the 6-12 month band showed the highest decline share; length showed no clear monotonic association with traffic.

[Validated model (holdout)]
  On a client-grouped holdout, the Random Forest ranks likely-decliners at precision@20 ~0.70 versus the 0.67 base rate (AUC ~0.61).

[Decision-support (no cause)]
  These pages look worth an editor reviewing first, because the pattern in this data is associated with decline - not a promise that refreshing them will restore rankings.

Skeptic check - words that exceeded the evidence in the original:
  'predict': only allowed as 'predict ranks/flags ...' (model), or 'associated with' (data), never 'fixing causes'.
  'so an edito

**Final honest version:**

> "On a client-grouped holdout, the Random Forest **ranks** likely-decliners first at precision@20
> ~0.70 (base rate 0.67, AUC ~0.61). In this portfolio snapshot, pages off page one and pages in the
> 6-12 month age band **showed** higher decline. These pages **look worth an editor reviewing first**
> — observed, measured, and decision-support. No intervention was run, so nothing here claims that
> editing a page will improve its ranking."

Banned phrasings I avoided: *proves / causes / will increase / the algorithm rewards / we predicted
Google's algorithm.* Where the paper and I report a count, it carries its denominator; where a group
was *chosen* (the 365+ refreshed pages, page-one pages), the selection is named in the same sentence.

## Self-check

- [x] Section 1: two paper findings, each with label source + design, and my methodology questions
- [x] Section 2: Week-5 model re-run under naive-random AND honest grouped split (before/after) + leaky trap
- [x] Section 3: leakage audit re-run on the final feature set, base rate beside every metric
- [x] Section 4: boldest sentence rewritten down the observed/associated/ranked/decision-support ladder
- [x] No client names, URLs, provider values, or raw identifiers in any output (aggregates and paper stats only)
- [x] Claims use careful words: observed, measured, ranked/flagged, decision-support; banned words avoided
- [ ] The notebook runs top to bottom with no errors (Kernel → Restart & Run all)
- [ ] Committed to `work/notebooks/` — then submit repo URL on the ML-09 card